# Advanced Session: Evaluating Energy Above Hull
AI for Materials Science — Hands-on session 1

Section C of the in-class notebook built a phase diagram and printed `energy_above_hull` for every
entry in the table. This notebook is about that one number.

Energy above hull is the most used stability descriptor in computational materials screening. It
decides which candidates are carried forward and which are dropped, so it is worth knowing exactly
what it measures, how to compute it yourself, and where it stops being informative.

## Today's plan

### 0 · Setup
Install the libraries and enter your MP API key.

### A · What the number measures
Read the three energies the Materials Project reports for one composition, and work out which of
them can be compared across materials.

### B · Ranking structures at one composition
The LiFePO4 composition holds dozens of calculated structures. Sort their reported hull distances and count
how many fall within a chosen energy window above the hull.

### C · Computing the hull yourself
Build the Li-Fe-P-O hull from GGA/GGA+U entries and compare it with database values from that same calculation family.

### D · Reading a hull distance as a decomposition reaction
A phase above the hull has a lower-energy competing phase or mixture. Turn the returned weights
into a balanced reaction.

### E · The hull depends on the entry set
Remove one competitor and the same material changes its hull distance. This is the part that
matters when you screen.

---

Run the cells one at a time from the top.
Later cells reuse variables created in earlier ones, so skipping ahead gives you a "name is not
defined" error.
Lines starting with `##` inside the code are comments written for you; Python does not run them.

Unlike the preclass and in-class notebooks, this one reads no files from the course repository.
All materials data come from live queries, so there is no `git clone` step and **an MP API key is
required**.

This is optional self-study for students who have finished the in-class assignment. It is not
graded. You will reuse pandas, Composition and PhaseDiagram from the earlier notebooks.

Every example is worked through, so there is nothing to fill in and nothing to submit. At the end of
C, D and E, pause to compare your output with the checks in the text before moving on.

## 0. Setup

Two things to get in place: the libraries and your MP API key.

### 0-1. Install the libraries
`pymatgen` builds the convex hull and reads compositions, and `mp_api` queries the Materials
Project. NumPy, pandas and matplotlib come along with them.

In [ ]:
## A leading ! runs a terminal command instead of Python. -q keeps the install output quiet.
!pip install -q pymatgen mp_api

### 0-2. Import the libraries
Installing and importing are two different steps. Installing puts files on the machine; `import`
brings a tool into this notebook.

In [ ]:
## Standard Python tools for environment variables, file paths and hidden key entry.
import os
from pathlib import Path
from getpass import getpass

## np for numeric work, pd for tables, plt for figures.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Composition parses a chemical formula; PhaseDiagram builds the convex hull.
from pymatgen.core import Composition
from pymatgen.analysis.phase_diagram import PhaseDiagram

## The Materials Project query client.
from mp_api.client import MPRester

### 0-3. Set the output folder
The ranking figure in B-3 is saved here as a PNG. Tables are displayed in the notebook.

In [ ]:
## parents=True creates the outputs folder too; exist_ok=True makes a second run harmless.
OUTPUT = Path("outputs/03_advanced")
OUTPUT.mkdir(parents=True, exist_ok=True)

print("outputs:", OUTPUT.resolve())

### 0-4. Enter your MP API key
The data used throughout this notebook come from live Materials Project queries. There is no saved
copy to fall back on, so the key is required. You can get one for free from the
[MP account page](https://next-gen.materialsproject.org/api).

Treat the key like a password. Written into a code cell or a submitted file, it is exposed.
`getpass` takes the input without echoing it to the screen. If you would rather not type it every
time, set `MP_API_KEY` in the environment before starting the notebook and this cell will pick it up.

The cell prints the database version. This version lookup does not validate your key; the first
materials query in A-1 will do that. Write the version down: entries are added continuously,
so hull distances shift slightly between versions, and the version is part of any number you report.

In [ ]:
## Use the key from an environment variable if it is set; otherwise ask for it directly.
API_KEY = os.getenv("MP_API_KEY", "").strip()
if not API_KEY:
    API_KEY = getpass("Materials Project API key: ").strip()

## This notebook has no offline path, so stop here rather than failing halfway through.
if not API_KEY:
    raise ValueError("An API key is required. Run this cell again and enter your key.")

## Read the server database version. The query in A-1, not this lookup, checks the key.
with MPRester(API_KEY) as mpr:
    MP_DB_VERSION = mpr.db_version

print("MP database version:", MP_DB_VERSION)

## A. What energy above hull measures

The Materials Project reports several energies for every material and they are not interchangeable.
Section A is about telling them apart.

We use the LiFePO4 composition throughout the notebook. It is the cathode material from the preclass
notebook, and the database holds many calculated structures at that one composition, which is
exactly what we need.

### A-1. Query one composition
`summary.search` returns one document per material. `formula` matches on the reduced formula, so a
single call collects every structure at this composition. `fields` limits what comes back; without it
the response is far larger than we need.

The summary endpoint selects MP's preferred thermodynamic data for each material. That may be a
mixed GGA/GGA+U/r2SCAN result. We use these displayed values in A and B; in C we choose a single
calculation family explicitly for the numerical check.

In [ ]:
## The fields we need to put the three energies side by side.
SUMMARY_FIELDS = ["material_id", "formula_pretty", "symmetry",
                  "energy_per_atom", "formation_energy_per_atom",
                  "energy_above_hull", "is_stable"]

with MPRester(API_KEY) as mpr:
    lfp_docs = mpr.materials.summary.search(formula="LiFePO4", fields=SUMMARY_FIELDS)

print("Structures at the LiFePO4 composition:", len(lfp_docs))

### A-2. Three energies, one table
Every row below is a **different calculated structure at the same composition**.

In [ ]:
## One dictionary per document becomes one row of the table.
lfp_rows = []
for doc in lfp_docs:
    lfp_rows.append({
        "material_id": doc.material_id,
        ## symmetry.symbol is the space group. str() keeps it as plain text in the table.
        "spacegroup": str(doc.symmetry.symbol),
        "energy_eV_atom": doc.energy_per_atom,
        "formation_energy_eV_atom": doc.formation_energy_per_atom,
        "e_above_hull_eV_atom": doc.energy_above_hull,
        "is_stable": doc.is_stable,
    })

## Sorting puts the smallest reported hull distance in the first row.
lfp_summary = (pd.DataFrame(lfp_rows)
               .sort_values("e_above_hull_eV_atom")
               .reset_index(drop=True))

lfp_summary.head(10)

Read the three energy columns from left to right.

- `energy_eV_atom` — the **MP-corrected total DFT energy per atom**. The raw calculation energy is a
  separate field, `uncorrected_energy_per_atom`. These total energies depend on the calculation's
  reference, so do not rank different compositions by their total energy alone. Balanced energy
  differences require consistent calculation methods and corrections.
- `formation_energy_eV_atom` — energy relative to the elemental references, using the selected
  thermodynamic dataset. A negative value means formation from those elements is downhill. With
  consistent references it can be compared across compositions, but it does not establish stability
  against other compounds or mixtures.
- `e_above_hull_eV_atom` — the difference from the lowest-energy phase or mixture at the **same
  overall composition**. Zero means no included competitor is lower in energy. A positive value is
  the energy per atom released on reaching that mixture within the chosen 0 K model.

Hull distance is useful for screening because it includes competition with both individual phases
and composition-balanced mixtures.

### A-3. Checking the smallest reported hull distance
In the dataset used to prepare this example, LiFePO4 has one structure on the hull. Check the
minimum hull distance and the number of `is_stable` rows in your output. If a later database version
gives different counts, those are the results to report. Being the lowest-energy structure at a
composition does not, by itself, guarantee a hull distance of zero.

In [ ]:
## idxmin returns the row label of the minimum hull distance at this composition.
ground_state_row = lfp_summary.loc[lfp_summary["e_above_hull_eV_atom"].idxmin()]

print("Smallest reported hull distance:", ground_state_row["material_id"],
      "| spacegroup", ground_state_row["spacegroup"])
print("Its hull distance:", ground_state_row["e_above_hull_eV_atom"], "eV/atom")
print()
## True counts as 1 when summed, so this counts the flagged rows.
print("Rows flagged is_stable :", int(lfp_summary["is_stable"].sum()), "of", len(lfp_summary))
## Floating point rarely gives exactly 0, so compare against a small tolerance instead.
print("Rows below 1e-6 eV/atom:", int((lfp_summary["e_above_hull_eV_atom"] < 1e-6).sum()))

### A-4. What "the hull" is
Plot formation energy per atom against composition for every calculated phase in a chemical system.
The **convex hull** is the lower envelope of that cloud of points: the phases and phase mixtures that
no combination of other phases can undercut.

For an entry at composition $c$,

$$E_{\rm hull}(c) = E_{f}(c) - E_{f}^{\rm hull}(c)$$

where $E_f^{\rm hull}(c)$ is the height of the envelope at that composition. An entry sitting on the
envelope gives zero. We check this identity numerically in C-4.

Two consequences worth holding on to.

- Hull distance is a **vertical** distance at fixed composition. It is not a distance to the nearest
  stable compound.
- It is defined **relative to a set of competing phases**. Change the set and the number changes,
  which is what section E is about.

## B. Ranking structures at one composition

At a fixed composition and within one consistent thermodynamic model, all structures share the
same hull reference. Sorting by hull distance therefore also sorts their corrected energies. The
lowest structure is at zero only if it is stable against every included phase mixture.

The summary endpoint can choose different thermodynamic datasets for different rows. The table
below therefore ranks the **reported MP hull distances**; it does not by itself establish relative
polymorph energies within one method. In C, we select one family explicitly and compute that
consistent ranking from its entries.

### B-1. The whole ranking
Hull distances at this scale are easier to read in meV/atom than in eV/atom, so we add a converted
column.

In [ ]:
## copy() makes a table separate from lfp_summary, so adding a column does not touch the original.
lfp_ranking = lfp_summary[["material_id", "spacegroup", "e_above_hull_eV_atom"]].copy()
## 1 eV = 1000 meV. The whole column is converted in one operation.
lfp_ranking["e_above_hull_meV_atom"] = lfp_ranking["e_above_hull_eV_atom"] * 1000

print("Structures in the ranking:", len(lfp_ranking))
print("First structure          :", lfp_ranking.iloc[0]["material_id"])
print("Highest in this set      :",
      round(lfp_ranking["e_above_hull_meV_atom"].max(), 1), "meV/atom above the hull")

lfp_ranking.head(15)

### B-2. Choosing an energy window
Use **25 meV/atom** as an illustrative screening window. This is a value we choose to explore the
table, not a universal boundary between stable and unstable materials.

The count tells us how many candidates have small hull distances in this dataset. It does not
measure DFT uncertainty or the probability that one structure is the ground state. Closely spaced
candidates are worth checking for calculation convergence, sensitivity to the method, and
finite-temperature effects.

At 300 K, the thermal energy scale kBT is about 26 meV. That thermal scale alone is not a per-atom
stability criterion for bulk phases. Temperature-dependent ordering requires free-energy
differences, including entropy. We do not calculate those contributions here.

In [ ]:
## This is our chosen screening window in meV per atom, not a temperature calculation.
SCREENING_WINDOW_meV_ATOM = 25.0
print(f"Illustrative screening window: {SCREENING_WINDOW_meV_ATOM:.1f} meV/atom")
print()

## Count entries inside each window. 0.001 meV/atom is a small tolerance for "on the hull".
window_rows = []
for cut_meV in [0.001, 10, SCREENING_WINDOW_meV_ATOM, 50, 100, 200]:
    inside = int((lfp_ranking["e_above_hull_meV_atom"] <= cut_meV).sum())
    window_rows.append({"window_meV_atom": cut_meV,
                        "structures_inside": inside,
                        "fraction_of_set": inside / len(lfp_ranking)})

pd.DataFrame(window_rows)

### B-3. The shape of the ranking
Plot hull distance against rank so you can see the distribution without choosing histogram bins.
The dashed line is the same 25 meV/atom screening window used in B-2.

Look for the number of points below the line and the size of the gaps between successive points.
These describe the calculated energy landscape. They do not, on their own, establish the accuracy
of the ranking or which structures can be synthesized.

In [ ]:
## fig is the whole figure, ax is the plotting area.
fig, ax = plt.subplots(figsize=(7.2, 4.2))

## lfp_ranking is already sorted, so the row position is the rank.
ranks = np.arange(1, len(lfp_ranking) + 1)
ax.plot(ranks, lfp_ranking["e_above_hull_meV_atom"],
        marker="o", markersize=4, linewidth=1.2, color="#4a6fa5")
## axhline draws a horizontal reference line across the plot.
ax.axhline(SCREENING_WINDOW_meV_ATOM, linestyle="--", color="#b3261e", linewidth=1.4,
           label=f"Screening window: {SCREENING_WINDOW_meV_ATOM:.0f} meV/atom")

## The highest structure is far above the cluster near zero. symlog is a log scale that still
## accepts the value 0, so both ends of the range stay readable in one panel.
ax.set_yscale("symlog", linthresh=1)
## Cutting the axis at zero keeps the panel from reserving space for negative values.
ax.set_ylim(0, lfp_ranking["e_above_hull_meV_atom"].max() * 1.3)
ax.set_xlabel("Rank at the LiFePO4 composition")
ax.set_ylabel("Energy above hull (meV/atom)")
ax.set_title(f"LiFePO4 structures in MP {MP_DB_VERSION}")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT / "lfp_hull_distance_ranking.png", dpi=180)
plt.show()

## C. Computing the hull yourself

Now we build a hull from the underlying entries and check its values against the database. For this
check, both sides must use the **same calculation family**. We use GGA/GGA+U entries and compare
against GGA/GGA+U values from the thermo endpoint, rather than assuming the summary values in A
use that family.

This is useful when you later place a calculation of your own on a hull. Before interpreting a new
entry, first check that your setup reproduces known entries. Your own calculation would also need
compatible settings and corrections; adding an arbitrary raw DFT energy is not sufficient.

### C-1. Fetching every subsystem
A hull over Li-Fe-P-O needs more than the quaternary compounds. It needs **elemental references**
(Li, Fe, P, O2) and competing binary and ternary phases, such as Li2O, FePO4 and Li3PO4.
`get_entries_in_chemsys` returns entries from these subsystems as well as the quaternary system.

Two arguments matter here.

- `compatible_only=True` retains the compatibility corrections in the entries served by MP. These
  include corrections used to combine GGA and GGA+U energies. Setting it to False removes those
  adjustments and changes the energies used to build the hull.
- `additional_criteria={"thermo_types": ["GGA_GGA+U"]}` selects one calculation family. Without a
  selection, the endpoint can return entries from multiple thermodynamic datasets. That is not a
  ready-to-use single hull. In particular, mixed GGA/GGA+U/r2SCAN corrections need care when moving
  entries between chemical systems.

The query may take a few seconds. Wait for the entry count before running the next cell.

In [ ]:
## Reuse one thermo type for both the entries and the database reference in C-3.
PD_ELEMENTS = ["Li", "Fe", "P", "O"]
THERMO_TYPE = "GGA_GGA+U"

with MPRester(API_KEY) as mpr:
    pd_entries = mpr.get_entries_in_chemsys(
        PD_ELEMENTS, compatible_only=True,
        additional_criteria={"thermo_types": [THERMO_TYPE]})

print("Entries returned:", len(pd_entries))

## A set removes duplicates, so this lists each elemental reference once. All four must appear.
element_references = sorted({e.composition.reduced_formula
                             for e in pd_entries if e.composition.is_element})
print("Elemental references:", element_references)

## Count entries by how many elements they contain, to see the subsystems that came back.
element_counts = pd.Series([len(e.composition.elements) for e in pd_entries]).value_counts()
print("Entries by number of elements:", element_counts.sort_index().to_dict())

### C-2. What happens if you leave the subsystems out
It is worth seeing this failure rather than being told about it. Keeping only the four-element
compounds throws away the elemental references, and `PhaseDiagram` refuses to build.

In [ ]:
## Keep only entries that contain all four elements.
quaternary_only = [e for e in pd_entries if len(e.composition.elements) == 4]
print("Quaternary-only entries:", len(quaternary_only))

## try/except prints the error instead of stopping the notebook.
try:
    PhaseDiagram(quaternary_only)
    print("This line should not be reached.")
except ValueError as error:
    print("PhaseDiagram refused:", error)

Even a long entry list cannot define this hull without the elemental references. After those
references are included, the competing compounds also matter for obtaining the correct envelope.

### C-3. Building the hull and reproducing the database
`PhaseDiagram` takes the full entry list and computes the envelope. `get_e_above_hull` then places
an entry against it. We fetch the database reference from `thermo.search` with the same
`THERMO_TYPE` as the entries, and compare the LiFePO4 results.

One detail about identifiers: entries from `get_entries_in_chemsys` carry a calculation suffix, so
`mp-19017` comes back as `mp-19017-GGA+U`. Strip the suffix before matching against the
`material_id` values returned by `thermo.search`.

In [ ]:
phase_diagram = PhaseDiagram(pd_entries)
print("Stable phases on this hull:", len(phase_diagram.stable_entries))
print()

## Use thermo, not summary, so the reference has the same calculation family as our hull.
with MPRester(API_KEY) as mpr:
    lfp_thermo_docs = mpr.materials.thermo.search(
        formula="LiFePO4", thermo_types=[THERMO_TYPE], all_fields=False,
        fields=["material_id", "thermo_type", "energy_above_hull"])

## Each document becomes one reference row. The column name records that it is a DB value.
thermo_reference = pd.DataFrame([
    {"material_id": str(doc.material_id),
     "thermo_type": str(doc.thermo_type),
     "db_e_above_hull_eV_atom": float(doc.energy_above_hull)}
    for doc in lfp_thermo_docs
])
if thermo_reference.empty:
    raise LookupError(f"No LiFePO4 reference values returned for {THERMO_TYPE}.")

## split("-GGA")[0] turns mp-19017-GGA+U back into mp-19017.
own_rows = []
for entry in pd_entries:
    if entry.composition.reduced_formula == "LiFePO4":
        own_rows.append({
            "material_id": str(entry.entry_id).split("-GGA")[0],
            "own_e_above_hull_eV_atom": phase_diagram.get_e_above_hull(entry),
        })
own_hull = pd.DataFrame(own_rows)

## Match by material_id. validate checks that each ID occurs only once in each table.
comparison = thermo_reference.merge(own_hull, on="material_id", how="inner", validate="one_to_one")
comparison["difference_meV_atom"] = (
    (comparison["own_e_above_hull_eV_atom"] - comparison["db_e_above_hull_eV_atom"]) * 1000)
comparison = comparison.sort_values("db_e_above_hull_eV_atom").reset_index(drop=True)

print("Calculation family   :", THERMO_TYPE)
print("Rows matched         :", len(comparison), "of", len(thermo_reference), "DB rows")
print("Own LiFePO4 entries  :", len(own_hull))
if comparison.empty:
    raise LookupError("No material IDs matched. Inspect thermo_reference and own_hull before continuing.")
print("Largest disagreement :",
      round(comparison["difference_meV_atom"].abs().max(), 6), "meV/atom")

comparison.head(10)

Pause here and check the three counts: matched rows, database rows and your own LiFePO4 entries.
For a complete comparison they should agree. A difference below about 1 meV/atom shows agreement
**at the compositions tested**; it does not prove that every part of the quaternary hull is identical.

If the disagreement is larger, check the calculation family, corrections, included competitors and
database version. A difference from the summary table in A can also come from its use of a different
thermodynamic dataset; that alone would not mean your GGA/GGA+U hull is wrong.

### C-4. The identity from A-4, checked
`get_hull_energy_per_atom` returns the envelope height in the same corrected total-energy reference
as the entries. Subtracting it from the entry energy gives the same hull distance as subtracting
formation energies: the elemental-reference term cancels at fixed composition.

We select the runner-up from the entries used in this hull. The summary ranking in B may use another
calculation family, so we do not use it to choose an entry for this check.

In [ ]:
## Sort the LiFePO4 entries by corrected energy within the family used for this hull.
lfp_pd_entries = sorted(
    (e for e in pd_entries if e.composition.reduced_formula == "LiFePO4"),
    key=lambda e: e.energy_per_atom)
if len(lfp_pd_entries) < 2:
    raise LookupError("This comparison needs at least two LiFePO4 entries.")

## Index 1 is the second item because Python counts from zero.
runner_up = lfp_pd_entries[1]
runner_up_id = str(runner_up.entry_id).split("-GGA")[0]

hull_height = phase_diagram.get_hull_energy_per_atom(runner_up.composition)
by_hand = runner_up.energy_per_atom - hull_height
from_pymatgen = phase_diagram.get_e_above_hull(runner_up)

print("Entry                       :", runner_up_id)
print("Its energy per atom         :", round(runner_up.energy_per_atom, 6), "eV/atom")
print("Hull height at that composition:", round(hull_height, 6), "eV/atom")
print("Difference                  :", round(by_hand, 6), "eV/atom")
print("get_e_above_hull            :", round(from_pymatgen, 6), "eV/atom")
## np.isclose compares two floats while allowing for rounding error.
print("Identity holds              :", bool(np.isclose(by_hand, from_pymatgen)))

## D. Reading a hull distance as a decomposition reaction

A positive hull distance is not just a score. It says the phase is unstable **against a specific set
of products**, and pymatgen will tell you which ones.

`get_decomp_and_e_above_hull` returns both at once: the products with their weights, and the distance.
The products give the lowest-energy mixture predicted by this 0 K entry set. Actual experimental
products can differ because temperature, atmosphere and reaction kinetics also matter.

### D-1. Picking a target above the hull
We use Li4Fe2(PO4)3, another composition in the Li-Fe-P-O system. Rather
than hard-coding a material ID that could change between database versions, we take the
lowest-energy structure at that composition.

In [ ]:
TARGET_FORMULA = "Li4Fe2(PO4)3"

## Every structure the C-1 query returned at this composition.
target_candidates = [e for e in pd_entries
                     if e.composition.reduced_formula == TARGET_FORMULA]
if not target_candidates:
    raise LookupError(f"No entries at the {TARGET_FORMULA} composition in this query.")

## min with key picks the entry with the smallest hull distance.
target = min(target_candidates, key=phase_diagram.get_e_above_hull)
decomposition, target_e_hull = phase_diagram.get_decomp_and_e_above_hull(target)

print("Structures at this composition:", len(target_candidates))
print("Target entry     :", target.entry_id)
print("Hull distance    :", round(float(target_e_hull), 6), "eV/atom")
print("Number of products:", len(decomposition))

### D-2. Weights are atomic fractions, not reaction coefficients
The weights that come back say what fraction of the **atoms** ends up in each product. To write a
balanced reaction you have to convert them to coefficients per formula unit:

$$\text{coefficient} = \text{weight} \times \frac{N_{\rm target}}{N_{\rm product}}$$

where $N$ is the number of atoms in the reduced formula. The reduced formula Li4Fe2(PO4)3 contains 21 atoms, so a
product with 4 atoms per formula unit and a weight of 1/3 gets a coefficient of $1/3 \times 21/4$.

In [ ]:
## num_atoms on the reduced formula, not on the calculation cell, is what the conversion needs.
target_atom_count = Composition(TARGET_FORMULA).num_atoms

decomposition_rows = []
## .items() walks a dictionary as (key, value) pairs. Here the key is the product entry.
for product, weight in decomposition.items():
    product_formula = product.composition.reduced_formula
    product_atom_count = Composition(product_formula).num_atoms
    decomposition_rows.append({
        "product_formula": product_formula,
        "product_entry_id": str(product.entry_id).split("-GGA")[0],
        "atoms_per_formula_unit": product_atom_count,
        "atomic_fraction_weight": float(weight),
        "coefficient_per_formula_unit": (float(weight) * target_atom_count
                                         / product_atom_count),
    })

decomposition_table = (pd.DataFrame(decomposition_rows)
                       .sort_values("product_formula")
                       .reset_index(drop=True))

print(f"{TARGET_FORMULA} has {target_atom_count:.0f} atoms per formula unit")
print("Atomic fractions sum to:", round(decomposition_table["atomic_fraction_weight"].sum(), 6))

decomposition_table

The atomic fractions sum to 1 because every atom has to go somewhere. The coefficients do not sum to
anything in particular; they are the numbers you put in front of each product.

### D-3. Writing the reaction out and checking it
A decomposition reaction is only correct if every element balances. The cell below assembles the
reaction string and then checks the atom counts on both sides.

In [ ]:
## Build the right-hand side as text, leaving out a coefficient of exactly 1.
product_terms = []
for row in decomposition_table.itertuples():
    coefficient = row.coefficient_per_formula_unit
    prefix = "" if np.isclose(coefficient, 1.0) else f"{coefficient:.4g} "
    product_terms.append(f"{prefix}{row.product_formula}")

print(f"{TARGET_FORMULA} -> " + " + ".join(product_terms))
print()

## Sum coefficient x atom count for each element on the product side.
product_atoms = {}
for row in decomposition_table.itertuples():
    for element, amount in Composition(row.product_formula).get_el_amt_dict().items():
        ## get(element, 0.0) starts a newly seen element at zero instead of raising an error.
        product_atoms[element] = (product_atoms.get(element, 0.0)
                                  + row.coefficient_per_formula_unit * amount)

target_atoms = Composition(TARGET_FORMULA).get_el_amt_dict()
balance = pd.DataFrame([
    {"element": element,
     "left_side": target_atoms.get(element, 0.0),
     "right_side": product_atoms.get(element, 0.0)}
    for element in sorted(set(target_atoms) | set(product_atoms))
])
balance["balanced"] = np.isclose(balance["left_side"], balance["right_side"])

print("Every element balanced:", bool(balance["balanced"].all()))
balance

Before moving on, check D-2 and D-3: the atomic fractions should sum to 1, and every row of the
element-balance table should be True. The reaction coefficients need not sum to 1. If your checks
fail after an edit, check that you used reduced-formula atom counts for both target and products.

### D-4. Same composition, same hull reference
The LiFePO4 entries in this example share a hull reference. When the lowest-energy LiFePO4 structure
lies on that hull, the equilibrium product for a higher-energy LiFePO4 polymorph is that structure.
The next cell checks this for the entries we actually fetched.

This result is specific to the current hull. **Every polymorph also competes with mixtures of other
compositions.** If none of the structures at a composition lies on the hull, even its lowest-energy
polymorph decomposes into such a mixture. That is what we see for the Li4Fe2(PO4)3 target in D-1.

In [ ]:
lfp_decomposition_rows = []
for entry in pd_entries:
    ## continue skips to the next loop iteration, so only LiFePO4 entries are processed.
    if entry.composition.reduced_formula != "LiFePO4":
        continue
    products, e_hull = phase_diagram.get_decomp_and_e_above_hull(entry)
    lfp_decomposition_rows.append({
        "material_id": str(entry.entry_id).split("-GGA")[0],
        "e_above_hull_eV_atom": float(e_hull),
        "n_products": len(products),
        "products": " + ".join(sorted(p.composition.reduced_formula for p in products)),
    })

lfp_decomposition = (pd.DataFrame(lfp_decomposition_rows)
                     .sort_values("e_above_hull_eV_atom")
                     .reset_index(drop=True))

## nunique counts how many different values a column holds.
print("Structures checked   :", len(lfp_decomposition))
print("Distinct product sets:", lfp_decomposition["products"].nunique())
print()
print(lfp_decomposition["products"].value_counts().to_string())

lfp_decomposition.head(5)

## E. The hull depends on the entry set

This is the section that changes how you read a screening result.

`energy_above_hull` is not a property of a material. It is a property of a material **measured against
a list of competitors**. Change the list and the number changes, with no new calculation of the
material itself.

### E-1. Remove the ground state
Take the runner-up at the LiFePO4 composition from C-4, drop the ground state from the entry list,
and rebuild the hull.

In [ ]:
## The ground state at this composition is the LiFePO4 entry closest to the hull.
ground_state_entry = min((e for e in pd_entries
                          if e.composition.reduced_formula == "LiFePO4"),
                         key=phase_diagram.get_e_above_hull)

## "is not" compares identity, so this drops that one object and keeps every other entry.
without_ground_state = [e for e in pd_entries if e is not ground_state_entry]
hull_without_ground_state = PhaseDiagram(without_ground_state)

print("Removed:", str(ground_state_entry.entry_id).split("-GGA")[0])
print("Entries used :", len(pd_entries), "->", len(without_ground_state))
print("Stable phases:", len(phase_diagram.stable_entries), "->",
      len(hull_without_ground_state.stable_entries))
print()
print("Runner-up", runner_up_id, "hull distance")
print("  with the ground state present:",
      round(phase_diagram.get_e_above_hull(runner_up), 6), "eV/atom")
print("  with it removed              :",
      round(hull_without_ground_state.get_e_above_hull(runner_up), 6), "eV/atom")

Read the runner-up's two hull distances. If the second is zero, it became stable in the reduced
entry set without being recalculated. If it stays positive, a different phase mixture still lies
below it. Check the stable-phase counts too; removing one entry does not generally keep that count
fixed.

This illustrates a limitation of screening a new composition. A hull distance of zero means that
no included competitor beats it, but a lower-energy structure may not have been calculated yet.

### E-2. Remove a competitor instead
Now return to the full hull and remove every structure at the Li3Fe2(PO4)3 composition. This is one
of the D-2 products in the dataset used to prepare the notebook. Check your own product table:
if it is absent after a database update, removing it may leave the target unchanged.

We remove the whole composition so that another polymorph there cannot remain as a competitor.
Removing just one structure can change the hull too, but does not remove that composition entirely.

In [ ]:
COMPETITOR_FORMULA = "Li3Fe2(PO4)3"

## != keeps everything whose reduced formula is not the competitor.
without_competitor = [e for e in pd_entries
                      if e.composition.reduced_formula != COMPETITOR_FORMULA]
hull_without_competitor = PhaseDiagram(without_competitor)

new_products, new_e_hull = hull_without_competitor.get_decomp_and_e_above_hull(target)

print("Entries removed:", len(pd_entries) - len(without_competitor))
print()
print(f"{TARGET_FORMULA} hull distance")
print("  full entry set :", round(float(target_e_hull), 6), "eV/atom")
print("  competitor gone:", round(float(new_e_hull), 6), "eV/atom")
print()
print("Products, full entry set :",
      " + ".join(sorted(p.composition.reduced_formula for p in decomposition)))
print("Products, competitor gone:",
      " + ".join(sorted(p.composition.reduced_formula for p in new_products)))

Compare both the distances and the product lists. For this example's original dataset, removing
Li3Fe2(PO4)3 lowered the target's hull distance and changed the products. Use your displayed results
for the database version you queried; a competitor can also be removed without changing the target.

The direction has a useful check: **with the target energy fixed and the target retained, removing
competitors cannot increase its hull distance; adding competitors cannot decrease it.** A missing
competitor can therefore make a candidate appear more stable. If your removal calculation increases
the distance beyond rounding error, check that you kept the same target and energy corrections.

### E-3. What this means when you screen
Keep these points in mind when you next use a hull-distance filter.

- **Keep methods and references consistent.** Check the calculation family, corrections, database
  version and competitor coverage. Across chemical systems, consistently calculated meV/atom values
  can be compared as decomposition driving forces, but the same cutoff need not imply the same
  synthesizability or the same prediction accuracy in each chemistry.
- **Report the database version and the entry-selection rule.** Your result describes a particular
  set of available calculations. The entry set can change in a later release.
- **Treat small gaps as candidates for further checks.** B counted structures inside energy windows;
  it did not estimate an error bar or a finite-temperature stability boundary.
- **Separate thermodynamic driving force from synthesizability.** Hull distance alone does not tell
  you the decomposition barrier or which synthesis route will work.

The hull here uses corrected 0 K electronic energies, with the solid pV contribution treated as
negligible. We have not calculated temperature-dependent free energies, atmosphere effects or
reaction rates. Those are additional questions, even when the numerical hull checks pass.